# Toy Agent Comparison

This notebook now uses the actual repo agents and the repo evaluation path on `TopKEnv`.
Read the outputs here as a mechanism study, not as the main paper benchmark pipeline.

The main checks in this version are:
- masked vs unmasked partial-feedback SPO
- full vs partial vs phased comparisons
- flip-set and contested-arm diagnostics from the repo evaluation utilities
- completed multi-arm runs instead of the interrupted larger-arm cell

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from lets_plot import LetsPlot

from notebooks.toy_agent_comparison_helpers import (
    default_main_configs,
    default_scale_configs,
    plot_diagnostics,
    plot_crossing_trace,
    plot_prediction_profiles,
    plot_regret,
    print_regret_summary,
    run_crossing_trace_suite,
    run_scale_cases,
    run_suite,
)

LetsPlot.setup_html()
SMOOTH_WINDOW = 31


In [2]:
W_2ARM = np.array([[1.0], [2.5]], dtype=float)
main_configs = default_main_configs(include_mse=True)
display(pd.DataFrame([{'label': label, **cfg} for label, cfg in main_configs]))


,label,agent_name,full_feedback,mask_grad,phased
0,BayesSPO (full),BayesianSPO,True,True,NaN
1,"BayesSPO (partial, masked)",BayesianSPO,False,True,NaN
2,"BayesSPO (partial, unmasked)",BayesianSPO,False,False,NaN
3,BayesSPO-Phased (partial),BayesianSPO,False,True,True
4,BayesianMSE (full),BayesianMSE,True,False,NaN
5,BayesianMSE (partial),BayesianMSE,False,False,NaN
6,BayesianMSE-Phased (partial),BayesianMSE,False,False,True


## 2-arm linear

This is the easy, well-specified case. If the bottleneck story is right, full feedback should dominate, but partial-feedback SPO should not necessarily collapse.

In [3]:
linear_rounds, linear_summary, linear_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='linear',
    capture_agents=True,
)
display(linear_summary)
print_regret_summary(linear_summary)
display(plot_regret(linear_rounds, '2-arm linear regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(linear_rounds, '2-arm linear diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(linear_artifacts, W_2ARM, cost_fn='linear'))


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesianMSE (full),0.084917,0.162061,0.9968,0.0616,0.080280,0.064317,0.995190
1,BayesSPO (full),0.085556,0.182919,0.9968,0.0624,0.114060,0.062771,0.995190
2,BayesSPO-Phased (partial),4.385252,3.781904,0.9924,0.0696,0.196606,0.082809,0.986373
3,"BayesSPO (partial, masked)",4.420352,3.832447,0.9884,0.0848,0.181962,0.084191,0.978357
4,BayesianMSE (partial),4.677812,3.930668,0.9780,0.1400,0.157763,0.087330,0.957515
5,"BayesSPO (partial, unmasked)",4.678863,3.956861,0.9788,0.1504,0.191067,0.084191,0.959118
6,BayesianMSE-Phased (partial),4.700412,3.760618,0.9804,0.1344,0.157185,0.085392,0.962325


BayesianMSE (full)            : regret = 0.1 +/- 0.2
BayesSPO (full)               : regret = 0.1 +/- 0.2
BayesSPO-Phased (partial)     : regret = 4.4 +/- 3.8
BayesSPO (partial, masked)    : regret = 4.4 +/- 3.8
BayesianMSE (partial)         : regret = 4.7 +/- 3.9
BayesSPO (partial, unmasked)  : regret = 4.7 +/- 4.0
BayesianMSE-Phased (partial)  : regret = 4.7 +/- 3.8


## 2-arm sigmoid

This is the misspecified case. The direct masked vs unmasked SPO comparison is the key check for the claim that the main failure mode is contaminated imputed comparisons rather than gradient masking alone.

In [4]:
sigmoid_rounds, sigmoid_summary, sigmoid_artifacts = run_suite(
    W_true=W_2ARM,
    k=1,
    configs=main_configs,
    n_rounds=500,
    n_seeds=5,
    hidden_dim=8,
    cost_fn='sigmoid',
    capture_agents=True,
)
display(sigmoid_summary)
print_regret_summary(sigmoid_summary)
display(plot_regret(sigmoid_rounds, '2-arm sigmoid regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(sigmoid_rounds, '2-arm sigmoid diagnostics', smooth_window=SMOOTH_WINDOW))
display(plot_prediction_profiles(sigmoid_artifacts, W_2ARM, cost_fn='sigmoid'))


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),0.026956,0.044898,0.9932,1.1424,0.478719,0.065626,0.987976
1,BayesianMSE (full),0.657968,0.599523,0.9492,1.0184,0.256620,0.084227,0.899800
2,BayesSPO-Phased (partial),14.152125,1.399933,0.8748,1.7328,0.136197,0.107381,0.750701
3,BayesianMSE-Phased (partial),14.617129,1.235877,0.8660,1.7440,0.141845,0.107045,0.733066
4,"BayesSPO (partial, unmasked)",29.198794,2.010569,0.6384,1.9880,0.419371,0.085133,0.276954
5,"BayesSPO (partial, masked)",29.526775,1.595603,0.7428,1.9672,0.257630,0.104723,0.486172
6,BayesianMSE (partial),30.231549,1.476768,0.7512,1.9720,0.275169,0.102815,0.503006


BayesSPO (full)               : regret = 0.0 +/- 0.0
BayesianMSE (full)            : regret = 0.7 +/- 0.6
BayesSPO-Phased (partial)     : regret = 14.2 +/- 1.4
BayesianMSE-Phased (partial)  : regret = 14.6 +/- 1.2
BayesSPO (partial, unmasked)  : regret = 29.2 +/- 2.0
BayesSPO (partial, masked)    : regret = 29.5 +/- 1.6
BayesianMSE (partial)         : regret = 30.2 +/- 1.5


### Crossing Trace And Frozen Test Regret

This focuses on the SPO family only. The crossing trace tracks the estimated decision boundary over training, and the frozen test summary evaluates the final predictor greedily without UCB exploration.

In [5]:
spo_trace_configs = [item for item in main_configs if 'BayesSPO' in item[0]]
sigmoid_trace, sigmoid_frozen_summary, sigmoid_true_crossing = run_crossing_trace_suite(
    W_true=W_2ARM,
    k=1,
    configs=spo_trace_configs,
    n_rounds=500,
    hidden_dim=8,
    cost_fn='sigmoid',
    checkpoint_every=10,
)
display(sigmoid_frozen_summary)
display(plot_crossing_trace(sigmoid_trace, sigmoid_true_crossing, '2-arm sigmoid crossing trace'))


,agent,final_crossing,crossing_error,frozen_test_regret_mean,frozen_test_regret_std,frozen_decision_rate
0,BayesSPO (full),0.008649,0.008649,0.000002,0.000055,0.9990
1,BayesSPO-Phased (partial),0.068546,0.068546,0.000361,0.002455,0.9745
2,"BayesSPO (partial, unmasked)",-0.262830,0.262830,0.004845,0.016437,0.8975
3,"BayesSPO (partial, masked)",-0.608440,0.608440,0.021503,0.046337,0.7740


## Multi-arm scaling

These cases use the same repo agents and diagnostics, but on larger fixed-weight `TopKEnv` settings.

In [6]:
scale_configs = default_scale_configs()
scale_cases = [
    dict(label='4 arms, k=2', n_arms=4, k=2, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=42),
    dict(label='8 arms, k=4', n_arms=8, k=4, context_dim=5, hidden_dim=8, n_rounds=500, n_seeds=3, weight_seed=43),
    # dict(label='16 arms, k=4', n_arms=16, k=4, context_dim=5, hidden_dim=16, n_rounds=500, n_seeds=3, weight_seed=44),
]
scale_results = run_scale_cases(scale_cases, scale_configs, cost_fn='sigmoid')
for label, case in scale_results.items():
    print(label)
    display(case['summary'])
    print_regret_summary(case['summary'])
    print()


4 arms, k=2


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),1.797498,0.860302,0.840667,1.068000,0.487786,0.160804,0.886840
1,BayesianMSE (full),2.679430,1.749776,0.788667,1.276000,0.345053,0.163676,0.846493
2,BayesianMSE-Phased (partial),16.199360,1.662562,0.574000,2.489333,0.195112,0.214842,0.616940
3,BayesSPO-Phased (partial),16.329840,2.765846,0.563333,2.504000,0.191483,0.215374,0.600107
4,"BayesSPO (partial, masked)",39.178924,0.857635,0.321333,3.285333,0.349237,0.211872,0.316873
5,BayesianMSE (partial),39.241666,0.440782,0.304000,3.341333,0.364319,0.211583,0.310728
6,"BayesSPO (partial, unmasked)",42.082513,0.758382,0.235333,3.340000,0.462552,0.207979,0.181135


BayesSPO (full)               : regret = 1.8 +/- 0.9
BayesianMSE (full)            : regret = 2.7 +/- 1.7
BayesianMSE-Phased (partial)  : regret = 16.2 +/- 1.7
BayesSPO-Phased (partial)     : regret = 16.3 +/- 2.8
BayesSPO (partial, masked)    : regret = 39.2 +/- 0.9
BayesianMSE (partial)         : regret = 39.2 +/- 0.4
BayesSPO (partial, unmasked)  : regret = 42.1 +/- 0.8

8 arms, k=4


,agent,regret_mean,regret_std,decision_rate,mean_flip_size,mean_contested_mae,mean_contested_sigma,mean_rank_corr
0,BayesSPO (full),6.593660,0.671865,0.554000,2.777333,0.489827,0.161445,0.860992
1,BayesianMSE (full),7.952117,1.932672,0.512667,2.998667,0.350261,0.162077,0.840395
2,BayesSPO-Phased (partial),35.280924,1.520341,0.305333,4.948000,0.205694,0.215870,0.668003
3,BayesianMSE-Phased (partial),35.587685,0.759013,0.308667,5.058667,0.212337,0.215845,0.665219
4,"BayesSPO (partial, masked)",85.232631,4.542510,0.075333,6.780000,0.375449,0.209769,0.357127
5,BayesianMSE (partial),88.350935,4.159687,0.065333,6.802667,0.390687,0.209703,0.324713
6,"BayesSPO (partial, unmasked)",95.657400,1.033698,0.040667,6.825333,0.486220,0.207513,0.202469


BayesSPO (full)               : regret = 6.6 +/- 0.7
BayesianMSE (full)            : regret = 8.0 +/- 1.9
BayesSPO-Phased (partial)     : regret = 35.3 +/- 1.5
BayesianMSE-Phased (partial)  : regret = 35.6 +/- 0.8
BayesSPO (partial, masked)    : regret = 85.2 +/- 4.5
BayesianMSE (partial)         : regret = 88.4 +/- 4.2
BayesSPO (partial, unmasked)  : regret = 95.7 +/- 1.0



In [7]:
focus_case = '8 arms, k=4'
display(plot_regret(scale_results[focus_case]['rounds'], f'{focus_case} regret', smooth_window=SMOOTH_WINDOW))
display(plot_diagnostics(scale_results[focus_case]['rounds'], f'{focus_case} diagnostics', smooth_window=SMOOTH_WINDOW))
